# Data Preparation e primeiro modelo — tráfego de rede IoT

Projeto: IoT Malware Network Traffic Dataset (CIC-YNU-2026).

Objetivo: preparar os dados para classificar `MalwareFamily` a partir das características de tráfego.

**Cuidados importantes**
- O dataset é obtido diretamente do Kaggle; não guardar o CSV no repositório GitHub.
- `Hash` é usado para separar grupos, não como variável de entrada do modelo.
- Todas as transformações aprendidas (imputação, seleção de colunas e escalabilidade) são ajustadas apenas no treino.
- A divisão é feita por `Hash` para evitar que linhas do mesmo grupo apareçam em treino, validação e teste.
- Famílias com poucos hashes podem não ter exemplos suficientes para uma avaliação fiável. O notebook apresenta essa limitação em vez de esconder os resultados.


## 1. Instalar/importar bibliotecas e obter o dataset

In [ ]:
%pip install -q kagglehub scikit-learn pandas numpy matplotlib

import glob
import os
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
TARGET = "MalwareFamily"
GROUP = "Hash"

dataset_path = kagglehub.dataset_download(
    "raneemababneh/iot-malware-network-traffic-dataset-cic-ynu-2026"
)
csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError("Não foi encontrado nenhum CSV no dataset descarregado.")
CSV_PATH = csv_files[0]
df = pd.read_csv(CSV_PATH)

print("Ficheiro:", CSV_PATH)
print("Dimensão inicial:", df.shape)


## 2. Data Cleaning — valores infinitos, duplicados e colunas desnecessárias

In [ ]:
if TARGET not in df.columns or GROUP not in df.columns:
    raise KeyError(f"O dataset tem de conter as colunas {TARGET!r} e {GROUP!r}.")

print("Classes originais:")
display(df[TARGET].value_counts())

# Converter infinitos em valores em falta.
numeric_cols = df.select_dtypes(include=np.number).columns
inf_count = int(np.isinf(df[numeric_cols]).sum().sum())
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Remover linhas duplicadas exatas.
duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# Remover colunas constantes (exceto o alvo e o identificador de grupo).
constant_cols = [
    c for c in df.columns
    if c not in [TARGET, GROUP] and df[c].nunique(dropna=True) <= 1
]

# Remover variáveis identificadas como redundantes no EDA, se existirem.
redundant_cols = [c for c in ["Tot size", "LLC", "Arch"] if c in df.columns]
drop_features = list(dict.fromkeys(constant_cols + redundant_cols))

print("Infinitos convertidos em NaN:", inf_count)
print("Duplicados removidos:", duplicates_before)
print("Colunas constantes:", constant_cols)
print("Colunas redundantes/identificadoras removidas das features:", redundant_cols)
print("Dimensão após limpeza:", df.shape)
print("Valores em falta por coluna:")
display(df.isna().sum()[lambda x: x > 0])


## 3. Verificar classes e hashes por família

In [ ]:
# Contar hashes por família e agrupar famílias com menos de 10 hashes.
# Estas famílias não têm grupos suficientes para uma avaliação separada fiável.
hashes_per_class_original = df.groupby(TARGET)[GROUP].nunique().sort_values()
rare_classes = hashes_per_class_original[hashes_per_class_original < 10].index.tolist()
print("Famílias com menos de 10 hashes:", rare_classes)

if rare_classes:
    df[TARGET] = df[TARGET].replace({name: "Other_rare" for name in rare_classes})

# Um grupo Hash deve corresponder a uma única família para que a divisão estratificada por grupos faça sentido.
families_per_hash = df.groupby(GROUP)[TARGET].nunique()
multi_family_hashes = int((families_per_hash > 1).sum())
print("Hashes associados a mais de uma família:", multi_family_hashes)

hash_class = (
    df.groupby(GROUP)[TARGET]
      .agg(lambda s: s.mode().iloc[0])
      .rename(TARGET)
      .reset_index()
)

hashes_per_class = hash_class[TARGET].value_counts().sort_values()
rows_per_class = df[TARGET].value_counts()
class_summary = pd.concat(
    [hashes_per_class.rename("hashes"), rows_per_class.rename("linhas")],
    axis=1
).fillna(0).astype(int)
display(class_summary.sort_values("hashes"))

print("Número de hashes únicos:", df[GROUP].nunique())


## 4. Separar treino, validação e teste por Hash

In [ ]:
# Separar os HASHES, não as linhas individuais. Assim o mesmo Hash não atravessa conjuntos.
# A estratificação é feita ao nível de cada Hash, usando a família predominante do grupo.
# Com classes muito raras, a proporção 80/10/10 pode não ser exata.

group_labels = hash_class[TARGET]
group_ids = hash_class[GROUP]

split1 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_group_pos, temp_group_pos = next(split1.split(group_ids.to_frame(), group_labels))

train_hashes = set(group_ids.iloc[train_group_pos])
temp_hashes = group_ids.iloc[temp_group_pos].reset_index(drop=True)
temp_labels = group_labels.iloc[temp_group_pos].reset_index(drop=True)

split2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED + 1)
val_pos, test_pos = next(split2.split(temp_hashes.to_frame(), temp_labels))
val_hashes = set(temp_hashes.iloc[val_pos])
test_hashes = set(temp_hashes.iloc[test_pos])

train_df = df[df[GROUP].isin(train_hashes)].copy()
val_df = df[df[GROUP].isin(val_hashes)].copy()
test_df = df[df[GROUP].isin(test_hashes)].copy()

# Não usar o alvo nem Hash como variável de entrada.
excluded = [TARGET, GROUP] + drop_features
feature_cols = [c for c in df.columns if c not in excluded]
X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_val, y_val = val_df[feature_cols], val_df[TARGET]
X_test, y_test = test_df[feature_cols], test_df[TARGET]

print(f"Treino: {len(train_df):,} linhas ({len(train_df)/len(df):.1%})")
print(f"Validação: {len(val_df):,} linhas ({len(val_df)/len(df):.1%})")
print(f"Teste: {len(test_df):,} linhas ({len(test_df)/len(df):.1%})")


## 5. Validar a separação e inspecionar o desequilíbrio

In [ ]:
train_hashes_actual = set(train_df[GROUP].unique())
val_hashes_actual = set(val_df[GROUP].unique())
test_hashes_actual = set(test_df[GROUP].unique())

print("Hashes partilhados treino/validação:", len(train_hashes_actual & val_hashes_actual))
print("Hashes partilhados treino/teste:", len(train_hashes_actual & test_hashes_actual))
print("Hashes partilhados validação/teste:", len(val_hashes_actual & test_hashes_actual))

assert not (train_hashes_actual & val_hashes_actual)
assert not (train_hashes_actual & test_hashes_actual)
assert not (val_hashes_actual & test_hashes_actual)

distribution = pd.concat([
    y_train.value_counts().rename("treino"),
    y_val.value_counts().rename("validação"),
    y_test.value_counts().rename("teste")
], axis=1).fillna(0).astype(int)
display(distribution)

print("Classes ausentes no treino:", sorted(set(y_val.unique()) - set(y_train.unique())))
print("Nota: se uma classe tiver poucos Hashes, pode não surgir em todos os conjuntos; isso limita a avaliação dessa classe.")


## 6. Missing values, outliers e feature scaling

In [ ]:
# Converter os dados de entrada para numéricos, quando possível.
# O dataset fornecido é maioritariamente numérico; colunas não numéricas são sinalizadas.
non_numeric = X_train.select_dtypes(exclude=np.number).columns.tolist()
print("Colunas não numéricas:", non_numeric)

# Aplicar log1p apenas a colunas numéricas contínuas não negativas e enviesadas.
# Não aplicar log a variáveis binárias nem a identificadores.
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
skew_train = X_train[numeric_features].skew(numeric_only=True)
log_cols = [
    c for c in numeric_features
    if X_train[c].notna().any()
    and abs(skew_train.get(c, 0)) > 1
    and X_train[c].min(skipna=True) >= 0
    and X_train[c].nunique(dropna=True) > 2
]
print(f"Colunas com distribuição enviesada transformadas com log1p ({len(log_cols)}):")
print(log_cols)

# O log1p comprime valores extremos sem apagar linhas. A mediana trata NaN.
# O scaler é incluído para permitir reutilizar o pipeline noutros modelos;
# para Random Forest, a escala não costuma ser necessária.
log_transform = ColumnTransformer(
    transformers=[("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one"), log_cols)],
    remainder="passthrough",
    verbose_feature_names_out=False
)

preprocessor = Pipeline(steps=[
    ("log", log_transform),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Features usadas:", len(feature_cols))
print("Valores em falta no treino:", int(X_train.isna().sum().sum()))


## 7. Primeiro modelo de referência: Random Forest

In [ ]:
# Primeiro modelo de referência. O teste fica reservado para a avaliação final.
# Se a execução demorar demasiado, reduzir n_estimators para 30.
model = RandomForestClassifier(
    n_estimators=60,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1
)

baseline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", model)
])

baseline.fit(X_train, y_train)
val_pred = baseline.predict(X_val)

print("F1 macro (validação):", round(f1_score(y_val, val_pred, average="macro", zero_division=0), 4))
print("F1 weighted (validação):", round(f1_score(y_val, val_pred, average="weighted", zero_division=0), 4))
print(classification_report(y_val, val_pred, digits=3, zero_division=0))


## 8. Guardar resultados para comparação

In [ ]:
from sklearn.metrics import accuracy_score

baseline_results = {
    "modelo": "Random Forest",
    "accuracy": accuracy_score(y_val, val_pred),
    "f1_macro": f1_score(y_val, val_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_val, val_pred, average="weighted", zero_division=0)
}
results_df = pd.DataFrame([baseline_results])
display(results_df)


## 9. Registar decisões para o relatório

- **Limpeza:** infinitos convertidos em valores em falta; duplicados exatos removidos.
- **Features:** `Hash` excluído das entradas e usado apenas para manter grupos separados; colunas constantes/redundantes removidas.
- **Missing values:** imputação pela mediana dentro do pipeline, ajustada no treino.
- **Outliers:** transformação `log1p` em colunas numéricas enviesadas, sem remover automaticamente linhas que podem representar tráfego malicioso.
- **Splitting:** treino/validação/teste por `Hash` para reduzir data leakage. As percentagens reais são impressas porque a divisão por grupos pode não produzir exatamente 80/10/10.
- **Feature scaling:** `StandardScaler` no pipeline; pode ser útil para modelos sensíveis à escala.
- **Imbalanced data:** `class_weight="balanced_subsample"` no Random Forest; a distribuição das classes e o F1 macro são reportados.
- **Limitação:** famílias representadas por poucos hashes podem não ter suporte suficiente para uma avaliação fiável. Não interpretar métricas dessas classes sem verificar o suporte.


## 10. Próxima fase: seleção do modelo

Este Random Forest é apenas um **modelo de referência**, não uma escolha final. Na fase seguinte vamos comparar pelo menos:
1. Random Forest;
2. Extra Trees;
3. um modelo adicional adequado ao tempo e aos recursos disponíveis.

A comparação deve usar a mesma divisão treino/validação e dar prioridade ao **F1 macro**, à matriz de confusão e ao recall das classes de malware. O conjunto de teste só deve ser usado depois de escolhermos o modelo final.

# Passo 8 — Treinar e avaliar o Random Forest

Este passo utiliza o conjunto de treino para ajustar o modelo e o conjunto de validação para avaliar o seu desempenho. O conjunto de teste permanece reservado para a avaliação final.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import pandas as pd

# Confirmar que as variáveis do Passo 7 estão disponíveis.
required_vars = ['X_train', 'y_train', 'X_val', 'y_val', 'preprocessor']
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise NameError(f'Executa primeiro as células anteriores. Faltam: {missing_vars}')

# Random Forest: primeiro modelo a avaliar.
rf_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
y_val_pred = rf_model.predict(X_val)

metrics_rf = pd.DataFrame([{
    'Modelo': 'Random Forest',
    'Accuracy': accuracy_score(y_val, y_val_pred),
    'Precision macro': precision_score(y_val, y_val_pred, average='macro', zero_division=0),
    'Recall macro': recall_score(y_val, y_val_pred, average='macro', zero_division=0),
    'F1 macro': f1_score(y_val, y_val_pred, average='macro', zero_division=0),
    'F1 weighted': f1_score(y_val, y_val_pred, average='weighted', zero_division=0)
}])

display(metrics_rf.round(4))

## 8.1 — Relatório de classificação

O relatório mostra precision, recall, F1-score e número de exemplos por classe. Observa especialmente o recall das classes de malware e o F1 macro.

In [ ]:
print(classification_report(y_val, y_val_pred, digits=4, zero_division=0))

## 8.2 — Matriz de confusão

A matriz de confusão permite ver quais famílias são confundidas entre si. A diagonal representa as previsões corretas; os restantes valores representam erros.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_predictions(
    y_val, y_val_pred,
    xticks_rotation=90,
    normalize=None,
    values_format='d',
    ax=ax,
    colorbar=False
)
ax.set_title('Matriz de confusão — Random Forest (validação)')
plt.tight_layout()
plt.show()

## 8.3 — Guardar o modelo de referência

Este ficheiro é uma versão de referência treinada com o conjunto de treino. Não é ainda o modelo final: depois da seleção e afinação, voltaremos a treinar e avaliar o modelo escolhido.

In [ ]:
import joblib

joblib.dump(rf_model, 'random_forest_baseline.joblib')
print('Modelo de referência guardado em random_forest_baseline.joblib')

## 9 — Ajustar o Random Forest e comparar resultados

Esta célula treina uma segunda versão do Random Forest e compara-a com o modelo de referência usando apenas o conjunto de validação. O conjunto de teste continua reservado para a avaliação final.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import pandas as pd

# Confirmar que as células anteriores foram executadas.
required_vars = ["preprocessor", "X_train", "y_train", "X_val", "y_val", "rf_model", "y_val_pred"]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise NameError(
        f"Faltam estas variáveis: {missing_vars}. Executa o notebook desde o início até ao Passo 8."
    )

# Treinar a versão ajustada do Random Forest.
rf_ajustado = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=30,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("A treinar o Random Forest ajustado. Pode demorar alguns minutos...")
rf_ajustado.fit(X_train, y_train)
y_val_pred_ajustado = rf_ajustado.predict(X_val)

# Comparar o modelo original e o ajustado na validação.
def calcular_metricas(nome, y_real, y_previsto):
    return {
        "Modelo": nome,
        "Accuracy": accuracy_score(y_real, y_previsto),
        "Precision macro": precision_score(y_real, y_previsto, average="macro", zero_division=0),
        "Recall macro": recall_score(y_real, y_previsto, average="macro", zero_division=0),
        "F1 macro": f1_score(y_real, y_previsto, average="macro", zero_division=0),
        "F1 weighted": f1_score(y_real, y_previsto, average="weighted", zero_division=0)
    }

comparacao = pd.DataFrame([
    calcular_metricas("Random Forest inicial", y_val, y_val_pred),
    calcular_metricas("Random Forest ajustado", y_val, y_val_pred_ajustado)
])

display(comparacao.round(4))

print("\\nRelatório de classificação — Random Forest ajustado:")
print(classification_report(y_val, y_val_pred_ajustado, digits=4, zero_division=0))

# Guardar a nova versão para eventual utilização futura.
import joblib
joblib.dump(rf_ajustado, "random_forest_ajustado.joblib")
print("\\nModelo ajustado guardado em random_forest_ajustado.joblib")


## Passo 10 — Terceira configuração do Random Forest (baixo consumo de RAM)

Esta versão usa menos árvores, limita a profundidade e treina cada árvore com uma amostra dos dados. Foi simplificada para reduzir o consumo de memória no Google Colab. O resultado é experimental; o conjunto de teste continua reservado para a avaliação final.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import pandas as pd
import joblib
import gc

required_vars = ['preprocessor', 'X_train', 'y_train', 'X_val', 'y_val', 'y_val_pred', 'y_val_pred_ajustado']
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise NameError(f'Faltam variáveis: {missing_vars}. Executa as células anteriores até ao Passo 9.')

# Configuração mais leve: menos árvores, profundidade limitada e amostras por árvore reduzidas.
rf_teste3 = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=60,
        max_depth=18,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features='sqrt',
        max_samples=0.6,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=2
    ))
])
print('A treinar a configuração de baixo consumo de RAM...')
rf_teste3.fit(X_train, y_train)
y_val_pred_teste3 = rf_teste3.predict(X_val)

def metricas_modelo(nome, y_real, y_previsto):
    return {
        'Modelo': nome,
        'Accuracy': accuracy_score(y_real, y_previsto),
        'Precision macro': precision_score(y_real, y_previsto, average='macro', zero_division=0),
        'Recall macro': recall_score(y_real, y_previsto, average='macro', zero_division=0),
        'F1 macro': f1_score(y_real, y_previsto, average='macro', zero_division=0),
        'F1 weighted': f1_score(y_real, y_previsto, average='weighted', zero_division=0)
    }

comparacao_3_modelos = pd.DataFrame([
    metricas_modelo('Random Forest inicial', y_val, y_val_pred),
    metricas_modelo('Random Forest ajustado', y_val, y_val_pred_ajustado),
    metricas_modelo('Random Forest teste 3 (baixo RAM)', y_val, y_val_pred_teste3)
])
print('\nComparação na validação:')
display(comparacao_3_modelos.round(4))
print('\nRelatório de classificação — terceira configuração:')
print(classification_report(y_val, y_val_pred_teste3, digits=4, zero_division=0))

joblib.dump(rf_teste3, 'random_forest_teste3_baixa_ram.joblib')
print('Modelo guardado em random_forest_teste3_baixa_ram.joblib')

# Libertar o objeto do modelo depois de o guardar; as previsões e métricas ficam disponíveis.
del rf_teste3
gc.collect()


## Passo 11 — Otimização adicional: comparar com Extra Trees

O Random Forest ajustado teve o melhor F1 macro entre as três configurações anteriores. Esta etapa testa um modelo adicional, Extra Trees, com limites de memória. Todos os modelos são comparados na mesma validação; o conjunto de teste continua reservado para a avaliação final.

**Importante:** executa o notebook desde o início, por ordem, antes desta célula.

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
import pandas as pd
import matplotlib.pyplot as plt
import joblib, gc

et_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', ExtraTreesClassifier(
        n_estimators=80,
        max_depth=24,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,
        max_samples=0.7,
        class_weight='balanced',
        random_state=42,
        n_jobs=2
    ))
])

print('A treinar Extra Trees com limites de memória...')
et_model.fit(X_train, y_train)
y_val_pred_et = et_model.predict(X_val)

def metricas_comparacao(nome, y_real, y_previsto):
    return {
        'Modelo': nome,
        'Accuracy': accuracy_score(y_real, y_previsto),
        'Precision macro': precision_score(y_real, y_previsto, average='macro', zero_division=0),
        'Recall macro': recall_score(y_real, y_previsto, average='macro', zero_division=0),
        'F1 macro': f1_score(y_real, y_previsto, average='macro', zero_division=0),
        'F1 weighted': f1_score(y_real, y_previsto, average='weighted', zero_division=0)
    }

resultados_otimizacao = pd.DataFrame([
    metricas_comparacao('Random Forest inicial', y_val, y_val_pred),
    metricas_comparacao('Random Forest ajustado', y_val, y_val_pred_ajustado),
    metricas_comparacao('Random Forest baixo RAM', y_val, y_val_pred_teste3),
    metricas_comparacao('Extra Trees baixo RAM', y_val, y_val_pred_et)
]).sort_values('F1 macro', ascending=False).reset_index(drop=True)

print('Comparação dos modelos na validação (ordenada por F1 macro):')
display(resultados_otimizacao.round(4))
print('\nRelatório por classe — Extra Trees:')
print(classification_report(y_val, y_val_pred_et, digits=4, zero_division=0))

fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_predictions(
    y_val, y_val_pred_et, xticks_rotation=90,
    values_format='d', ax=ax, colorbar=False
)
ax.set_title('Matriz de confusão — Extra Trees (validação)')
plt.tight_layout()
plt.show()

# Guardar o candidato Extra Trees para comparação futura; não o declarar automaticamente vencedor.
joblib.dump(et_model, 'extra_trees_baixa_ram.joblib')
print('Modelo guardado em extra_trees_baixa_ram.joblib')
del et_model
gc.collect()

## Passo 12 — Escolher o candidato sem usar o teste

Escolhe o modelo pela validação, dando prioridade ao F1 macro e verificando o recall das classes de malware e a matriz de confusão. Não escolhas apenas pela accuracy. Depois de fixar a configuração, treina-a novamente conforme necessário e usa o conjunto de teste uma única vez para a avaliação final. Regista também o tempo de treino e o consumo de memória, se possível.

## Passo 11 — Pesquisa controlada de hiperparâmetros do Random Forest

O Random Forest ajustado foi o melhor candidato até agora pelo F1 macro. Esta etapa testa poucas configurações para limitar o consumo de RAM. A pesquisa usa uma amostra estratificada do treino e uma amostra da validação; por isso, os resultados servem para selecionar candidatos e devem ser confirmados depois na validação completa. O conjunto de teste não é utilizado nesta pesquisa.

Executa o notebook desde o início. Se o teu ambiente tiver pouca memória, reduz `MAX_TRAIN_TUNING` e `MAX_VAL_TUNING`.

In [ ]:
import gc
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, classification_report
import joblib

# Limites para reduzir o consumo de memória durante a pesquisa.
MAX_TRAIN_TUNING = 100_000
MAX_VAL_TUNING = 30_000

if len(X_train) > MAX_TRAIN_TUNING:
    X_tune, _, y_tune, _ = train_test_split(
        X_train, y_train, train_size=MAX_TRAIN_TUNING,
        random_state=42, stratify=y_train
    )
else:
    X_tune, y_tune = X_train, y_train

if len(X_val) > MAX_VAL_TUNING:
    X_val_tune, _, y_val_tune, _ = train_test_split(
        X_val, y_val, train_size=MAX_VAL_TUNING,
        random_state=43, stratify=y_val
    )
else:
    X_val_tune, y_val_tune = X_val, y_val

print(f'Amostra de treino para otimização: {len(X_tune):,} linhas')
print(f'Amostra de validação para otimização: {len(X_val_tune):,} linhas')

# Pequena pesquisa manual. Não é uma pesquisa exaustiva.
configuracoes_rf = [
    {'n_estimators': 80, 'max_depth': 24, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'},
    {'n_estimators': 100, 'max_depth': 30, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': 'balanced'},
    {'n_estimators': 80, 'max_depth': 32, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 0.5, 'class_weight': 'balanced_subsample'},
    {'n_estimators': 100, 'max_depth': 24, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'},
    {'n_estimators': 80, 'max_depth': 18, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.7, 'class_weight': 'balanced_subsample'},
]

resultados_tuning = []
melhor_f1_macro = -1
melhor_config_rf = None

for i, params in enumerate(configuracoes_rf, start=1):
    print(f'\nA testar configuração {i}/{len(configuracoes_rf)}: {params}')
    candidato = Pipeline(steps=[
        ('preprocessing', preprocessor),
        ('classifier', RandomForestClassifier(
            **params,
            max_samples=0.7,
            random_state=42,
            n_jobs=2
        ))
    ])
    candidato.fit(X_tune, y_tune)
    pred = candidato.predict(X_val_tune)
    f1m = f1_score(y_val_tune, pred, average='macro', zero_division=0)
    resultados_tuning.append({
        'Configuração': i,
        'F1 macro': f1m,
        'Recall macro': recall_score(y_val_tune, pred, average='macro', zero_division=0),
        'Precision macro': precision_score(y_val_tune, pred, average='macro', zero_division=0),
        'Accuracy': accuracy_score(y_val_tune, pred),
        'Parâmetros': str(params)
    })
    if f1m > melhor_f1_macro:
        melhor_f1_macro = f1m
        melhor_config_rf = params.copy()
    del candidato, pred
    gc.collect()

tabela_tuning = pd.DataFrame(resultados_tuning).sort_values('F1 macro', ascending=False).reset_index(drop=True)
display(tabela_tuning.round(4))
print('\nMelhor configuração na amostra de validação:')
print(melhor_config_rf)
print('F1 macro da amostra:', round(melhor_f1_macro, 4))

# Guardar apenas os parâmetros escolhidos, para reproduzir a experiência.
with open('melhores_parametros_random_forest.json', 'w', encoding='utf-8') as f:
    import json
    json.dump(melhor_config_rf, f, ensure_ascii=False, indent=2)
print('Parâmetros guardados em melhores_parametros_random_forest.json')

## Passo 12 — Confirmar o candidato na validação completa

Depois de analisar a tabela anterior, executa esta célula para treinar a configuração escolhida na amostra de treino e avaliá-la na validação completa. Este resultado é mais comparável aos modelos anteriores. Se o F1 macro não melhorar, mantém o Random Forest ajustado anterior como candidato principal.

In [ ]:
rf_otimizado = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(
        **melhor_config_rf,
        max_samples=0.7,
        random_state=42,
        n_jobs=2
    ))
])

print('A treinar o candidato escolhido na amostra de treino...')
rf_otimizado.fit(X_tune, y_tune)
y_val_pred_otimizado = rf_otimizado.predict(X_val)

print('Métricas na validação completa:')
print('Accuracy:', round(accuracy_score(y_val, y_val_pred_otimizado), 4))
print('Precision macro:', round(precision_score(y_val, y_val_pred_otimizado, average='macro', zero_division=0), 4))
print('Recall macro:', round(recall_score(y_val, y_val_pred_otimizado, average='macro', zero_division=0), 4))
print('F1 macro:', round(f1_score(y_val, y_val_pred_otimizado, average='macro', zero_division=0), 4))
print('F1 weighted:', round(f1_score(y_val, y_val_pred_otimizado, average='weighted', zero_division=0), 4))
print('\nRelatório por classe:')
print(classification_report(y_val, y_val_pred_otimizado, digits=4, zero_division=0))

joblib.dump(rf_otimizado, 'random_forest_otimizado_candidato.joblib')
print('Candidato guardado em random_forest_otimizado_candidato.joblib')

## Passo 13 — Matriz de confusão do Random Forest otimizado

Executa esta célula depois do Passo 12, quando `y_val` e `y_val_pred_otimizado` já estiverem definidos.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_val_pred_otimizado,
    xticks_rotation=45,
    values_format='d',
    ax=ax,
    colorbar=False
)
ax.set_title('Matriz de confusão — Random Forest otimizado')
plt.tight_layout()
plt.show()